# Convolutional Neural Networks (CNNs), Explained Simply
Filters, feature maps, pooling, and a CNN vs a plain network, in PyTorch. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The data
3,000 small grey pictures (28 x 28 pixels) of four hand-drawn shapes: circle, square, triangle and cross, in random places and sizes, with noise.

### 3,000 shape pictures
- `shapes.npz`: a compressed NumPy file holding the images and labels
- `urlretrieve`: download the file into Colab, unless it is already there
- `images.shape`: 3000 pictures, each 28 x 28 pixels
- `labels`: 0 circle, 1 square, 2 triangle, 3 cross

In [ ]:
import numpy as np, os, urllib.request

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "cnns/shapes.npz")
if not os.path.exists("shapes.npz"):
    urllib.request.urlretrieve(url, "shapes.npz")
data = np.load("shapes.npz")
images, labels = data["images"], data["labels"]
names = ["circle", "square", "triangle", "cross"]
print(images.shape, labels[:8])

### Look at some shapes
- `ax.imshow`: draw a grid of numbers as a picture: 0 black, 255 white

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, lab in zip(axes, images, labels):
    ax.imshow(img, cmap="gray")
    ax.set_title(names[lab]); ax.axis("off")
plt.show()

## 2. Convolution by hand
A **filter** is a small grid of weights (here 3 x 3). Slide it over the image; at each position multiply the overlapping numbers and add them up. The result is a **feature map**. This filter, -1 0 1 in every row, lights up where dark changes to bright: a vertical edge.

### One filter, by hand
- `img`: a 5 x 5 image: dark on the left, bright on the right
- `filt`: a vertical edge filter
- `.view(1, 1, 5, 5)`: PyTorch wants (pictures, channels, height, width)
- `F.conv2d`: slide the filter over the image
- `out`: 27 where the filter covers the edge, 0 on the flat bright area

In [ ]:
import torch
import torch.nn.functional as F

img = torch.tensor([[0, 0, 9, 9, 9]] * 5, dtype=torch.float32)
filt = torch.tensor([[-1, 0, 1]] * 3, dtype=torch.float32)
out = F.conv2d(img.view(1, 1, 5, 5), filt.view(1, 1, 3, 3))
print(img)
print(out.squeeze())

### Max pooling, by hand
- `F.max_pool2d(..., 2)`: split into 2 x 2 blocks and keep the biggest number in each
- `the output`: a 2 x 2 grid: 4, 2, 2, 5

In [ ]:
grid = torch.tensor([[1, 3, 2, 0],
                     [4, 2, 1, 1],
                     [0, 1, 5, 2],
                     [2, 0, 3, 4]], dtype=torch.float32)
print(F.max_pool2d(grid.view(1, 1, 4, 4), 2).squeeze())

### Edge filters on a real shape
- `vert.T`: turn the filter on its side: a horizontal edge filter
- `padding=1`: add a border of zeros so the feature map stays 28 x 28
- `F.relu`: keep only the positive responses
- `the picture`: one map lights up the square's left and right sides, the other its top and bottom

In [ ]:
x = torch.tensor(images[1] / 255, dtype=torch.float32).view(1, 1, 28, 28)
vert = torch.tensor([[-1, 0, 1]] * 3, dtype=torch.float32)
filters = torch.stack([vert, vert.T]).unsqueeze(1)
maps = F.relu(F.conv2d(x, filters, padding=1))
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, im, t in zip(axes, [x[0, 0], maps[0, 0], maps[0, 1]],
                     ["image", "vertical edges", "horizontal edges"]):
    ax.imshow(im, cmap="gray"); ax.set_title(t); ax.axis("off")
plt.show()

## 3. Prepare the data
Scale pixels to 0 to 1, add a channel dimension, and split: 2,400 to train, 600 to test.

### Scale, convert, split
- `.unsqueeze(1)`: add a channel: grey pictures have 1 (colour pictures have 3)
- `X[:2400]`: the pictures are already shuffled, so the first 2,400 train

In [ ]:
import torch.nn as nn

X = torch.tensor(images / 255, dtype=torch.float32).unsqueeze(1)
y = torch.tensor(labels)
X_tr, y_tr, X_te, y_te = X[:2400], y[:2400], X[2400:], y[2400:]
print(X_tr.shape, X_te.shape)

### A training function
- `mini-batches of 64`: shuffle, then 64 pictures per step
- `zero_grad / backward / step`: clear, backpropagate, update
- `returns`: training accuracy and test accuracy

In [ ]:
def fit(model, epochs=15):
    torch.manual_seed(0)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(X_tr))
        for i in range(0, len(X_tr), 64):
            idx = order[i:i + 64]
            optimizer.zero_grad()
            loss_fn(model(X_tr[idx]), y_tr[idx]).backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        acc = lambda X, y: (model(X).argmax(1) == y).float().mean().item()
        return acc(X_tr, y_tr), acc(X_te, y_te)

## 4. A plain network first
Flatten each picture into 784 numbers and use ordinary layers. Every pixel gets its own weight to every neuron.

### Plain network
- `nn.Flatten()`: 28 x 28 grid -> a row of 784 numbers
- `nn.Linear(784, 128)`: 128 neurons, each with 784 weights
- `the output`: perfect on training pictures, much worse on new ones: overfitting

In [ ]:
torch.manual_seed(0)
plain = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(),
                      nn.Linear(128, 4))
print("weights:", sum(p.numel() for p in plain.parameters()))
tr, te = fit(plain)
print(f"train accuracy {tr:.3f}, test accuracy {te:.3f}")

## 5. A CNN
Two rounds of **convolution -> ReLU -> max pooling**, then one ordinary layer to make the decision.

### Build the CNN
- `nn.Conv2d(1, 8, 3, padding=1)`: 8 filters of size 3 x 3, each learned
- `nn.MaxPool2d(2)`: halve the height and width
- `nn.Conv2d(8, 16, 3, ...)`: 16 filters, each looking at all 8 feature maps
- `16 * 7 * 7`: 16 maps of 7 x 7 after two poolings
- `the loop`: follow one picture through: 28 -> 14 -> 7

In [ ]:
torch.manual_seed(0)
cnn = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(16 * 7 * 7, 4))
print("weights:", sum(p.numel() for p in cnn.parameters()))
x = X_tr[:1]
for layer in cnn:
    x = layer(x)
    print(f"{layer.__class__.__name__:<10} {tuple(x.shape)}")

### Train the CNN
- `fit(cnn)`: exactly the same training as the plain network

In [ ]:
tr, te = fit(cnn)
print(f"train accuracy {tr:.3f}, test accuracy {te:.3f}")

### What the CNN learned
- `cnn[0].weight`: the 8 learned 3 x 3 filters (top row)
- `cnn[0](X_te[:1])`: their feature maps for one test picture (bottom row)
- `the picture`: each filter picks out different edges and directions

In [ ]:
with torch.no_grad():
    maps = F.relu(cnn[0](X_te[:1]))[0]
fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for k in range(8):
    axes[0, k].imshow(cnn[0].weight[k, 0].detach(), cmap="gray")
    axes[1, k].imshow(maps[k], cmap="gray")
    axes[0, k].axis("off"); axes[1, k].axis("off")
plt.show()

## 6. Mini project: name the shapes
Ask the CNN about some test pictures.

### Name eight test shapes
- `guess`: the CNN's answer for 8 test pictures (numbers 12 to 19)
- `color=...`: green if right, red if wrong

In [ ]:
with torch.no_grad():
    guess = cnn(X_te[12:20]).argmax(1)
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, g, t in zip(axes, X_te[12:20], guess, y_te[12:20]):
    ax.imshow(img[0], cmap="gray")
    ax.set_title(names[g], color="green" if g == t else "red")
    ax.axis("off")
plt.show()

## Try it yourself
1. Work out the convolution by hand at the top-right position. Then check it with the code.
2. Change the CNN's first layer from 8 filters to 2. How does the accuracy change?
3. Train the plain network for 30 epochs. Does the gap close?